## How to run in Colab
1. Open this notebook on GitHub and click **Open in Colab** (or go to colab.research.google.com → File → Open notebook → GitHub and paste `simonlivson/crypto-market-explorer`).
2. Click **Runtime → Run all**.
3. I don't use any API key. If the CoinGecko or Coin Metrics API is down or rate-limited, my code automatically loads the saved CSV files from my GitHub repo instead, so every cell still runs.

# Crypto Market Explorer: Cycles, Altseasons and Coins That Break Away From Bitcoin

Most of the crypto market moves together with Bitcoin. But in some phases altcoins outperform (an "altseason"), and some coins break away from Bitcoin for months, like Hyperliquid and Zcash in 2025.

In this project I want to find out in which market phases diversification in crypto is real, and which coins are just "Bitcoin in disguise". My story has two parts:

- **Part A, the big picture:** Bitcoin and Ethereum daily history since 2013/2015, to study the 4-year cycle and altseasons.
- **Part B, inside the current phase:** the last 365 days for 12 coins, to see who follows Bitcoin and who breaks away.

Before I can analyze anything, I need data. This first section collects it.

# 1. Data Collection

In this section I collect three datasets:

| Dataset | Source | What it gives me |
|---|---|---|
| `coins_365d.csv` | CoinGecko API (free, no key) | daily price, market cap and volume for 12 coins, last 365 days |
| `btc_eth_history.csv` | Coin Metrics Community API (free, no key) | daily BTC and ETH prices since 2013 / 2015 |
| `coin_categories.csv` | my own file | the theme of each coin (Layer 1, Privacy, AI, ...) |

I save the raw API answers as JSON files and the cleaned tables as CSV files in my `data/` folder. If an API fails, I fall back to the CSV copies on GitHub.

## 1.1 Importing my libraries

I only need a few libraries here:
- `requests` sends HTTP requests to the APIs (like a browser does when it opens a page).
- `pandas` turns the answers into tables (DataFrames).
- `json` and `os` let me save the raw files to my folders.
- `time` lets me pause between requests so the free API doesn't block me.

I also find my `data/` folder. In Colab or from the repo root it is `data/`, but if I open the notebook from inside `notebooks/` (like in VS Code) it is `../data/`, so I check which one exists.

In [ ]:
import os
import json
import time

import requests
import pandas as pd

# my data folder: "data/" from the repo root or Colab, "../data/" from notebooks/
if os.path.isdir("data") or not os.path.isdir("../data"):
    DATA = "data/"
else:
    DATA = "../data/"
os.makedirs(DATA + "raw", exist_ok=True)
print("my data folder:", os.path.abspath(DATA))

# backup copies of my data on GitHub, in case an API fails
GITHUB_DATA = "https://raw.githubusercontent.com/simonlivson/crypto-market-explorer/main/data/"


## 1.2 Choosing my 12 coins

CoinGecko doesn't use tickers like "BTC". Every coin has its own **id** (for example `binancecoin` for BNB and `ripple` for XRP). I checked each id on CoinGecko, and I list them here with the ticker I'll use in my charts.

I picked large Layer 1 coins (BTC, ETH, SOL, ADA), BNB, XRP for payments, DOGE as a meme coin, LINK as an oracle, and four "theme" coins that had their own runs: HYPE (DEX/perps), ZEC and XMR (privacy) and TAO (AI).

In [ ]:
# CoinGecko id -> ticker
coins = {
    "bitcoin": "BTC",
    "ethereum": "ETH",
    "solana": "SOL",
    "binancecoin": "BNB",
    "ripple": "XRP",
    "dogecoin": "DOGE",
    "cardano": "ADA",
    "chainlink": "LINK",
    "hyperliquid": "HYPE",
    "zcash": "ZEC",
    "monero": "XMR",
    "bittensor": "TAO",
}
print(len(coins), "coins")


## 1.3 A safe way to call the API

The free CoinGecko API allows only a limited number of requests per minute. If I ask too fast, it answers with status code **429 (Too Many Requests)**.

So I write a small helper function `get_json()`:
- it tries the request up to 4 times,
- if the answer is 429 or a server error, it waits a bit longer each time (10, 20, 30 seconds) and tries again,
- if everything fails, it returns `None`, so my notebook doesn't crash.

In [ ]:
def get_json(url, params=None, tries=4):
    for attempt in range(1, tries + 1):
        try:
            response = requests.get(url, params=params, timeout=30)
            # 200 means everything is fine
            if response.status_code == 200:
                return response.json()
            print(f"  status {response.status_code}, attempt {attempt}")
        except requests.RequestException as error:
            print(f"  error: {error}, attempt {attempt}")
        # I wait longer after every failed try
        time.sleep(10 * attempt)
    return None


## 1.4 Fetching 365 days of data for each coin

For each coin I call the `market_chart` endpoint:

`https://api.coingecko.com/api/v3/coins/{id}/market_chart?vs_currency=usd&days=365&interval=daily`

The answer is JSON with three lists: `prices`, `market_caps` and `total_volumes`. Each item is a pair `[timestamp, value]`, where the timestamp is in milliseconds since 1970.

I save every answer as it is in `data/raw/<coin>.json`, so I always keep the original data. Between coins I pause 3 seconds to stay polite to the free API.

In [ ]:
url = "https://api.coingecko.com/api/v3/coins/{}/market_chart"
params = {"vs_currency": "usd", "days": 365, "interval": "daily"}

raw_data = {}
for coin_id in coins:
    print("fetching", coin_id)
    data = get_json(url.format(coin_id), params)
    if data is None:
        print("  failed:", coin_id)
        continue
    raw_data[coin_id] = data
    # I keep the original answer as a JSON file
    with open(f"{DATA}raw/{coin_id}.json", "w") as f:
        json.dump(data, f)
    time.sleep(3)

print(f"I got data for {len(raw_data)} of {len(coins)} coins")


## 1.5 Turning the JSON into one table

Now I turn the three lists of every coin into one **long-format** table. Long format means one row per coin per day, with these columns:

`date, coin, price, market_cap, volume`

This format is easy to filter (`df[df.coin == "BTC"]`), group (`groupby("coin")`) and pivot later.

Two small details:
- I convert the millisecond timestamps to dates with `pd.to_datetime(..., unit="ms")`.
- CoinGecko adds the current, unfinished day as the last point. I keep only one row per day and drop that last partial day, so every day is a complete one.

In [ ]:
tables = []
for coin_id, data in raw_data.items():
    # one small table per coin, one column per list
    prices = pd.DataFrame(data["prices"], columns=["ts", "price"])
    caps = pd.DataFrame(data["market_caps"], columns=["ts", "market_cap"])
    volumes = pd.DataFrame(data["total_volumes"], columns=["ts", "volume"])

    coin_df = prices.merge(caps, on="ts").merge(volumes, on="ts")
    coin_df["date"] = pd.to_datetime(coin_df["ts"], unit="ms").dt.normalize()
    coin_df["coin"] = coins[coin_id]
    tables.append(coin_df)

if tables:
    coins_df = pd.concat(tables, ignore_index=True)
    # one row per coin and day
    coins_df = coins_df.drop_duplicates(subset=["coin", "date"], keep="first")
    # the last day is still running, so I drop it
    coins_df = coins_df[coins_df["date"] < pd.Timestamp.today().normalize()]
    coins_df = coins_df[["date", "coin", "price", "market_cap", "volume"]]
else:
    coins_df = pd.DataFrame()

# if the API gave me nothing (or only part), I load my saved copy from GitHub
if len(raw_data) < len(coins):
    print("API failed, loading the saved CSV from GitHub")
    coins_df = pd.read_csv(GITHUB_DATA + "coins_365d.csv", parse_dates=["date"])
else:
    coins_df.to_csv(DATA + "coins_365d.csv", index=False)

coins_df.head()


### A first check of my coin data

I check how many rows each coin has and which dates they cover. A coin that is younger than one year (or listed later on CoinGecko) will have fewer rows. That matters later, because I can only compare coins over the days they all have.

In [ ]:
summary = coins_df.groupby("coin")["date"].agg(["count", "min", "max"])
summary.columns = ["days", "first_date", "last_date"]
print("total rows:", len(coins_df))
summary


## 1.6 Long-term Bitcoin and Ethereum history

CoinGecko's free tier only gives me 365 days of daily data, and one year is not enough to see a 4-year cycle. So for Part A I use the **Coin Metrics Community API**, which is free and needs no key:

`https://community-api.coinmetrics.io/v4/timeseries/asset-metrics?assets=btc,eth&metrics=PriceUSD&frequency=1d`

It returns a daily USD reference price for BTC (from 2010) and ETH (from August 2015). I start in 2013, as planned.

The answer comes in **pages**. Each page has a `next_page_url` if there's more data, so I keep following that link until there are no pages left.

In [ ]:
cm_url = "https://community-api.coinmetrics.io/v4/timeseries/asset-metrics"
cm_params = {
    "assets": "btc,eth",
    "metrics": "PriceUSD",
    "frequency": "1d",
    "start_time": "2013-01-01",
    "page_size": 10000,
}

rows = []
page = get_json(cm_url, cm_params)
while page is not None:
    rows.extend(page["data"])
    # I follow the next page until there is none
    next_url = page.get("next_page_url")
    if not next_url:
        break
    page = get_json(next_url)

if rows:
    with open(DATA + "raw/coinmetrics_btc_eth.json", "w") as f:
        json.dump(rows, f)
    history = pd.DataFrame(rows)
    history["date"] = pd.to_datetime(history["time"]).dt.tz_localize(None).dt.normalize()
    history["price"] = history["PriceUSD"].astype(float)
    # one column per coin: date, BTC, ETH
    history = history.pivot_table(index="date", columns="asset", values="price").reset_index()
    history.columns.name = None
    history = history.rename(columns={"btc": "BTC", "eth": "ETH"})
    history.to_csv(DATA + "btc_eth_history.csv", index=False)
else:
    print("API failed, loading the saved CSV from GitHub")
    history = pd.read_csv(GITHUB_DATA + "btc_eth_history.csv", parse_dates=["date"])

print(len(history), "days from", history["date"].min().date(), "to", history["date"].max().date())
print("first ETH price on", history.loc[history["ETH"].notna(), "date"].min().date())
history.tail()


Ethereum didn't exist before 2015, so its column is empty (NaN) for the early years. That's expected, not an error. I'll handle it in the cleaning section.

## 1.7 My coin categories

The last dataset is a small CSV I wrote by hand. It gives each coin a **theme**, so later I can check whether coins that break away from Bitcoin share a theme (hypothesis 7). I'll merge it with my price data using the `symbol` column.

In [ ]:
try:
    categories = pd.read_csv(DATA + "coin_categories.csv")
except FileNotFoundError:
    # in Colab the file isn't local, so I read it from GitHub
    categories = pd.read_csv(GITHUB_DATA + "coin_categories.csv")
categories


## 1.8 Summary of my data collection

I now have three datasets:
1. **coins_365d.csv**: 12 coins × about 365 days of price, market cap and volume from CoinGecko (raw JSON kept in `data/raw/`).
2. **btc_eth_history.csv**: daily BTC prices since 2013 and ETH since 2015 from Coin Metrics.
3. **coin_categories.csv**: the theme of every coin.

Next, in section 2, I clean the data: missing values, duplicates, data types and outliers.

# 2. Data Cleaning

Raw data is almost never ready for analysis. Before I trust any chart or test, I check my two price tables step by step:

1. **Data types**: are dates real dates and numbers real numbers?
2. **Missing values**: where are there gaps, and why?
3. **Duplicates**: does any coin have the same day twice?
4. **Sorting**: are the rows in time order? (needed for returns)
5. **Sanity checks**: no zero or negative prices, the same number of days for every coin.
6. **Outliers**: which days had extreme moves, and are they errors or real?
7. **Merging and new columns**: I add the coin themes and daily returns.

At the end I save two clean files: `coins_clean.csv` and `btc_eth_clean.csv`.

To make this section work on its own, I first reload both tables from my CSV files (or from GitHub if the local files aren't there).

In [ ]:
import numpy as np

def load_csv(name):
    # I use my local copy first, and the GitHub copy as a backup
    if os.path.exists(DATA + name):
        return pd.read_csv(DATA + name)
    return pd.read_csv(GITHUB_DATA + name)

coins_df = load_csv("coins_365d.csv")
history = load_csv("btc_eth_history.csv")
categories = load_csv("coin_categories.csv")
print(coins_df.shape, history.shape, categories.shape)


## 2.1 Data types

`info()` shows me every column, its type (`dtype`) and how many values are not empty.

When pandas reads a CSV, dates come in as plain text (`object`). Text dates can't be sorted by time or used for things like "day of the week", so I convert them with `pd.to_datetime()`. Prices, market caps and volumes should be decimal numbers (`float64`). I convert them with `pd.to_numeric(..., errors="coerce")`: if a value isn't a number, it becomes `NaN` instead of crashing, and I'll find it in the missing values step.

In [ ]:
coins_df.info()
print()
history.info()


In [ ]:
# text dates -> real dates
coins_df["date"] = pd.to_datetime(coins_df["date"])
history["date"] = pd.to_datetime(history["date"])

# all number columns as floats
for col in ["price", "market_cap", "volume"]:
    coins_df[col] = pd.to_numeric(coins_df[col], errors="coerce").astype(float)
for col in ["BTC", "ETH"]:
    history[col] = pd.to_numeric(history[col], errors="coerce").astype(float)

print(coins_df.dtypes, "\n")
print(history.dtypes)


## 2.2 Missing values

`isna().sum()` counts the empty cells in every column. For each gap I want to know *why* it's there before I decide what to do:

- In my **coin table**, a missing price or volume would mean CoinGecko had no data for that day.
- In my **history table**, I already know ETH is empty before **August 2015**, simply because Ethereum didn't exist yet. Those gaps are not errors. I keep those rows, because BTC has real prices there and I need them for the long cycle. When I analyze ETH, I'll just use the rows where ETH has a value.

In [ ]:
print("missing values in my coin table:")
print(coins_df.isna().sum(), "\n")
print("missing values in my history table:")
print(history.isna().sum())


In [ ]:
# where the ETH gap is: it should end right before Ethereum started trading
eth_missing = history[history["ETH"].isna()]
print("ETH missing from", eth_missing["date"].min().date(), "to", eth_missing["date"].max().date())
print("first ETH price:", history.loc[history["ETH"].notna(), "date"].min().date())

# any other gap in ETH after it started would be a real problem
first_eth = history.loc[history["ETH"].notna(), "date"].min()
print("ETH gaps after launch:", history.loc[history["date"] >= first_eth, "ETH"].isna().sum())


In [ ]:
# rows in my coin table with a missing number, if there are any
missing_rows = coins_df[coins_df[["price", "market_cap", "volume"]].isna().any(axis=1)]
print(len(missing_rows), "rows with a missing value")
missing_rows


**My decision:** I keep the ETH gap in the history table (it's real history, not an error). If my coin table has any rows with a missing price, I drop them, because a row without a price can't be used for returns. A missing market cap or volume alone I fill from the day before with `ffill()` inside each coin, since these numbers change slowly from one day to the next.

In [ ]:
# a row without a price is useless for me
coins_df = coins_df.dropna(subset=["price"])

# small gaps in market cap / volume: I take the value of the day before, per coin
coins_df = coins_df.sort_values(["coin", "date"])
coins_df[["market_cap", "volume"]] = coins_df.groupby("coin")[["market_cap", "volume"]].ffill()
print(coins_df.isna().sum())


## 2.3 Duplicates

A duplicate here means the same coin appears twice on the same day. That could happen if an API sends two points for one day. I check with `duplicated()` on the columns `date` + `coin` (and only `date` in the history table), and I keep the first row if I find any.

In [ ]:
dup_coins = coins_df.duplicated(subset=["date", "coin"]).sum()
dup_hist = history.duplicated(subset=["date"]).sum()
print("duplicates in my coin table:", dup_coins)
print("duplicates in my history table:", dup_hist)

coins_df = coins_df.drop_duplicates(subset=["date", "coin"], keep="first")
history = history.drop_duplicates(subset=["date"], keep="first")


## 2.4 Sorting

Daily returns compare each day with the day before, so the rows must be in time order. I sort my coin table by coin and then date, and my history table by date. `reset_index(drop=True)` gives me a clean 0, 1, 2, ... index again.

In [ ]:
coins_df = coins_df.sort_values(["coin", "date"]).reset_index(drop=True)
history = history.sort_values("date").reset_index(drop=True)
coins_df.head()


## 2.5 Sanity checks

Now I check that the numbers make sense:
- A price, market cap or volume can never be zero or negative.
- Every coin should have the same number of days (364), with no missing days in between.
- My history table should have one row for every calendar day, because crypto trades 7 days a week.

In [ ]:
# zero or negative numbers
for col in ["price", "market_cap", "volume"]:
    print(col, "<= 0:", (coins_df[col] <= 0).sum())
print("BTC <= 0:", (history["BTC"] <= 0).sum())
print("ETH <= 0:", (history["ETH"] <= 0).sum())


In [ ]:
# number of days and date range per coin
days = coins_df.groupby("coin")["date"].agg(["count", "min", "max"])
# days between first and last date + 1 = days there should be
days["expected"] = (days["max"] - days["min"]).dt.days + 1
days["missing_days"] = days["expected"] - days["count"]
days


In [ ]:
# the history table should have every calendar day
all_days = pd.date_range(history["date"].min(), history["date"].max(), freq="D")
print("days in my history:", len(history), "| calendar days:", len(all_days))
print("missing calendar days:", len(all_days.difference(history["date"])))


## 2.6 Merging the coin themes

My `coin_categories.csv` has the theme of each coin. I merge it onto my coin table with `merge()`, matching my `coin` column (the ticker, like "BTC") with the `symbol` column. I use `how="left"` so I keep all my price rows, and then I check that no coin ended up without a theme.

In [ ]:
categories = categories.rename(columns={"coin": "coingecko_id", "symbol": "coin"})
coins_df = coins_df.merge(categories, on="coin", how="left")

print("rows without a theme:", coins_df["theme"].isna().sum())
coins_df.groupby("theme")["coin"].unique()


## 2.7 New columns: daily returns

Prices are hard to compare between coins: BTC costs about \$100,000 and DOGE about \$0.20. So for most of my analysis I use **returns**, the change in percent from one day to the next:

- **daily_return** = today's price / yesterday's price - 1, made with `pct_change()`. A value of 0.05 means +5%.
- **log_return** = ln(today / yesterday). It's almost the same as the daily return for small moves, but log returns can simply be added up over time, which is why they're popular in finance.

I calculate them **per coin** with `groupby("coin")`, otherwise the first day of ETH would be compared with the last day of BTC. The first day of each coin has no "yesterday", so its return is `NaN`, which is expected.

In [ ]:
coins_df["daily_return"] = coins_df.groupby("coin")["price"].pct_change()
coins_df["log_return"] = np.log(coins_df["price"] / coins_df.groupby("coin")["price"].shift(1))

# the same for my long BTC and ETH history
for col in ["BTC", "ETH"]:
    history[col + "_return"] = history[col].pct_change(fill_method=None)
    history[col + "_log_return"] = np.log(history[col] / history[col].shift(1))

coins_df[["date", "coin", "price", "daily_return", "log_return"]].head()


I also add a few date columns that I'll use later for seasonality and cycles: the weekday, the month and the year.

In [ ]:
coins_df["weekday"] = coins_df["date"].dt.day_name()
coins_df["month"] = coins_df["date"].dt.month
history["weekday"] = history["date"].dt.day_name()
history["month"] = history["date"].dt.month
history["year"] = history["date"].dt.year


## 2.8 Outliers

An outlier is a value far away from the usual ones. I look for them in the **daily returns**, not in the prices, because a high price isn't strange, but a +40% day is.

I use the **IQR method** that I learned in the course:
- Q1 is the 25% value and Q3 the 75% value of a coin's returns, and IQR = Q3 - Q1.
- A return is an outlier if it's below Q1 - 1.5 × IQR or above Q3 + 1.5 × IQR.

I calculate the limits for **each coin separately**, because a normal day for a small coin like ZEC would be an extreme day for BTC.

**Important:** in crypto, these outliers are usually *real* moves (crashes, hype rallies, news), not data errors. They're exactly what my breakaway story is about. So I **don't delete them**. I only flag them in a new column `is_outlier`, so I can study them or leave them out on purpose later.

In [ ]:
def iqr_flag(returns):
    q1 = returns.quantile(0.25)
    q3 = returns.quantile(0.75)
    iqr = q3 - q1
    # True when the return is outside the IQR limits
    return (returns < q1 - 1.5 * iqr) | (returns > q3 + 1.5 * iqr)

coins_df["is_outlier"] = coins_df.groupby("coin")["daily_return"].transform(iqr_flag)

outliers = coins_df.groupby("coin")["is_outlier"].sum().sort_values(ascending=False)
print("outlier days per coin:")
outliers


In [ ]:
# my 10 most extreme days, up or down
coins_df["abs_return"] = coins_df["daily_return"].abs()
top_moves = coins_df.sort_values("abs_return", ascending=False).head(10)
coins_df = coins_df.drop(columns="abs_return")
top_moves[["date", "coin", "theme", "price", "daily_return", "volume"]]


As a second check, I also look at the **z-score**: how many standard deviations a return is away from the coin's average. Above 3 is very rare in a normal distribution. If crypto returns were "normal", only about 0.3% of days would pass that limit.

In [ ]:
def zscore(returns):
    return (returns - returns.mean()) / returns.std()

coins_df["z_score"] = coins_df.groupby("coin")["daily_return"].transform(zscore)
z_count = (coins_df["z_score"].abs() > 3).groupby(coins_df["coin"]).sum()
print("days with |z| > 3 per coin:")
z_count.sort_values(ascending=False)


In [ ]:
# the same IQR flag for my long BTC and ETH history
for col in ["BTC", "ETH"]:
    history[col + "_outlier"] = iqr_flag(history[col + "_return"])
    print(col, "outlier days:", history[col + "_outlier"].sum())

# my biggest BTC days since 2013
history.loc[history["BTC_return"].abs().sort_values(ascending=False).index[:5], ["date", "BTC", "BTC_return"]]


## 2.9 Saving my clean data

My data is now clean: correct types, no duplicates, sorted, checked, with themes, returns and outlier flags. I save both tables so the next sections (and my Streamlit dashboard) can start from the clean versions.

In [ ]:
coins_df.to_csv(DATA + "coins_clean.csv", index=False)
history.to_csv(DATA + "btc_eth_clean.csv", index=False)
print("coins_clean:", coins_df.shape)
print("btc_eth_clean:", history.shape)


## 2.10 Cleaning summary

- **Types:** I turned dates into `datetime` and all numbers into `float`.
- **Missing values:** the only gap is ETH before August 2015, which is real (Ethereum didn't exist yet), so I kept it.
- **Duplicates and sorting:** I removed any repeated coin-days and sorted everything by time.
- **Sanity checks:** no zero or negative values, and every coin covers the same days.
- **New columns:** theme, daily and log returns, weekday, month and year.
- **Outliers:** crypto has many extreme days. They're real market moves, so I kept them and flagged them with `is_outlier` and `z_score`.

Next, in section 3, I explore the data.